# TCC — pipeline no Kaggle
Ative **GPU** (Settings → Accelerator) e **Internet** (exige telefone verificado). O repositório é o centro do projeto; este notebook só executa.

In [ ]:
!git clone https://github.com/SEU_USUARIO/tcc-detector.git
%cd tcc-detector
!pip install -q -e ".[generate,dev]"
import torch; print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 0. Teste de fumaça (dados falsos, ~2 min)
Rode **primeiro**: confirma que o pipeline inteiro funciona neste ambiente.

In [ ]:
!make smoke

## 1. Testes de geração (10 imagens por gerador)
Anote tempo e memória. Se um gerador falhar, troque no `configs/generators.yaml`.
StyleGAN2 exige `git clone https://github.com/NVlabs/stylegan2-ada-pytorch external/stylegan2-ada-pytorch` e os `.pkl` preenchidos.

In [ ]:
for g in ['sd15', 'sdxl_turbo', 'biggan']:
    !python -m tccdet.generate.run --generator {g} --n 10
# !python -m tccdet.generate.run --generator stylegan2 --n 10

## 2. Geração em lote (rode como 'Save & Run All' para continuar com o navegador fechado)

In [ ]:
for g in ['stylegan2', 'sd15', 'biggan', 'sdxl_turbo']:
    !python -m tccdet.generate.run --generator {g}

## 3. Imagens reais
Anexe o dataset (Add Input) e ajuste `real.source_dir` em `configs/experiment.yaml`.

In [ ]:
!python -m tccdet.collect_real

## 4. Pipeline

In [ ]:
!make preprocess split shortcuts
!make train eval SEED=0
!make aggregate

## 5. Salvar resultados

In [ ]:
!zip -qr /kaggle/working/resultados.zip results data/splits data/processed/manifest.csv